In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [2]:
df=pd.read_csv("df_export.csv")

In [3]:
categorical_cols = ['Weather', 'RoadType', 'Landmarks']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, dtype=int), categorical_cols)
    ],
    remainder='passthrough'
)

In [4]:

features = [
    'latitude', 'longitude', 'Temperature', 'NumberofLanes', 'day', 'hour', 'minute',
    'minutes_since_midnight', 'sin_time', 'cos_time', 'te_geohash_time',
    'Weather', 'RoadType', 'Landmarks',
    'demand_lag_15m', 'demand_lag_30m', 'demand_rolling_mean_45m','demand_d48' # Your new features!
]

In [5]:
traffic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

In [6]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# Mute Optuna logging noise to keep your cell outputs clean
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Isolate the full training matrix
train_df = df[df['is_train'] == 1].copy()
X_train = train_df[features].copy()
y_train = train_df['demand'].reset_index(drop=True)

# Define your constant 5-Fold cross-validation splitter
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# --- Pre-process STREAM A: One-Hot Encoded (For LightGBM & XGBoost) ---
# Pre-transforming upfront saves a massive amount of computation time inside the CV loops
X_train_encoded = pd.DataFrame(traffic_pipeline.named_steps['preprocessor'].fit_transform(X_train))

# --- Pre-process STREAM B: Native String Arrays (For CatBoost - No OHE!) ---
categorical_cols = ['Weather', 'RoadType', 'Landmarks']
X_train_cat = X_train.copy()
for col in categorical_cols:
    X_train_cat[col] = X_train_cat[col].astype(str)

# Extract indices of categorical columns for CatBoost's engine
cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_cols]

# =========================================================================
# TUNING TRIALS (5-FOLD CROSS-VALIDATION DRIVEN)
# =========================================================================

# --- Trial 1: LightGBM ---
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 15.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 15.0, log=True),
        'verbose': -1, 'random_state': 42, 'n_jobs': -1
    }
    scores = []
    for tr_idx, va_idx in cv.split(X_train_encoded):
        X_tr, X_va = X_train_encoded.iloc[tr_idx], X_train_encoded.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        
        model = LGBMRegressor(**params).fit(X_tr, y_tr)
        preds = np.clip(model.predict(X_va), a_min=0, a_max=None)
        scores.append(r2_score(y_va, preds))
    return np.mean(scores)

print("Tuning LightGBM using 5-Fold Cross-Validation...")
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=10) # Set to 10 trials for speed, feel free to increase

# --- Trial 2: XGBoost ---
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2500, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'alpha': trial.suggest_float('alpha', 0.1, 15.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 15.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1
    }
    scores = []
    for tr_idx, va_idx in cv.split(X_train_encoded):
        X_tr, X_va = X_train_encoded.iloc[tr_idx], X_train_encoded.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        
        model = XGBRegressor(**params).fit(X_tr, y_tr)
        preds = np.clip(model.predict(X_va), a_min=0, a_max=None)
        scores.append(r2_score(y_va, preds))
    return np.mean(scores)

print("Tuning XGBoost using 5-Fold Cross-Validation...")
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=10)

# --- Trial 3: CatBoost (Bypasses One-Hot Encoding) ---
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'depth': trial.suggest_int('depth', 5, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 15.0),
        'random_seed': 42, 'verbose': 0, 'thread_count': -1
    }
    scores = []
    for tr_idx, va_idx in cv.split(X_train_cat):
        X_tr, X_va = X_train_cat.iloc[tr_idx], X_train_cat.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        
        model = CatBoostRegressor(**params).fit(X_tr, y_tr, cat_features=cat_features_idx)
        preds = np.clip(model.predict(X_va), a_min=0, a_max=None)
        scores.append(r2_score(y_va, preds))
    return np.mean(scores)

print("Tuning CatBoost Native using 5-Fold Cross-Validation...")
study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(objective_cat, n_trials=10)

print(f"\n🥇 Best 5-Fold CV Scores Discovered -> LGBM: {study_lgb.best_value:.4f} | XGB: {study_xgb.best_value:.4f} | CAT: {study_cat.best_value:.4f}")

c:\Users\Pulkit\Banglore_traffic_prediction\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tuning LightGBM using 5-Fold Cross-Validation...
Tuning XGBoost using 5-Fold Cross-Validation...
Tuning CatBoost Native using 5-Fold Cross-Validation...

🥇 Best 5-Fold CV Scores Discovered -> LGBM: 0.9543 | XGB: 0.9551 | CAT: 0.9555


In [10]:
from sklearn.base import clone

print("Generating clean Out-of-Fold (OOF) predictions with optimal hyperparameters...")

# Extract winning configurations
best_lgb = LGBMRegressor(**study_lgb.best_params, random_state=42, verbose=-1, n_jobs=-1)
best_xgb = XGBRegressor(**study_xgb.best_params, random_state=42, n_jobs=-1)
best_cat = CatBoostRegressor(**study_cat.best_params, random_state=42, verbose=0, thread_count=-1)

# Initialize arrays to hold validation predictions (OOF)
oof_preds_lgb = np.zeros(len(X_train))
oof_preds_xgb = np.zeros(len(X_train))
oof_preds_cat = np.zeros(len(X_train))

# Isolate competition test set features
X_test = df[df['is_train'] == 0][features].copy()
X_test_encoded = pd.DataFrame(traffic_pipeline.named_steps['preprocessor'].transform(X_test))

X_test_cat = X_test.copy()
for col in categorical_cols:
    X_test_cat[col] = X_test_cat[col].astype(str)

# Accumulators for cross-fold test inferences
test_preds_lgb = np.zeros(len(X_test))
test_preds_xgb = np.zeros(len(X_test))
test_preds_cat = np.zeros(len(X_test))

# Execute the 5-Fold CV Production Loop
for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_encoded), start=1):
    # Split Encoded Features (LightGBM / XGBoost)
    X_tr_enc, X_va_enc = X_train_encoded.iloc[tr_idx], X_train_encoded.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    
    # Split Native String Features (CatBoost)
    X_tr_c, X_va_c = X_train_cat.iloc[tr_idx], X_train_cat.iloc[va_idx]
    
    # Fit and Predict LightGBM
    model_lgb = clone(best_lgb).fit(X_tr_enc, y_tr)
    oof_preds_lgb[va_idx] = np.clip(model_lgb.predict(X_va_enc), a_min=0, a_max=None)
    test_preds_lgb += np.clip(model_lgb.predict(X_test_encoded), a_min=0, a_max=None) / cv.n_splits
    
    # Fit and Predict XGBoost
    model_xgb = clone(best_xgb).fit(X_tr_enc, y_tr)
    oof_preds_xgb[va_idx] = np.clip(model_xgb.predict(X_va_enc), a_min=0, a_max=None)
    test_preds_xgb += np.clip(model_xgb.predict(X_test_encoded), a_min=0, a_max=None) / cv.n_splits
    
    # Fit and Predict CatBoost
    model_cat = clone(best_cat).fit(X_tr_c, y_tr, cat_features=cat_features_idx)
    oof_preds_cat[va_idx] = np.clip(model_cat.predict(X_va_c), a_min=0, a_max=None)
    test_preds_cat += np.clip(model_cat.predict(X_test_cat), a_min=0, a_max=None) / cv.n_splits

    print(f"Fold {fold}/5 finalized.")

print("\n" + "="*50)
print(f"OOF LightGBM R2 Baseline : {r2_score(y_train, oof_preds_lgb):.4f}")
print(f"OOF XGBoost R2 Baseline  : {r2_score(y_train, oof_preds_xgb):.4f}")
print(f"OOF CatBoost R2 Baseline : {r2_score(y_train, oof_preds_cat):.4f}")
print("="*50)

Generating clean Out-of-Fold (OOF) predictions with optimal hyperparameters...
Fold 1/5 finalized.
Fold 2/5 finalized.
Fold 3/5 finalized.
Fold 4/5 finalized.
Fold 5/5 finalized.

OOF LightGBM R2 Baseline : 0.9544
OOF XGBoost R2 Baseline  : 0.9551
OOF CatBoost R2 Baseline : 0.9555


In [11]:
print("Running secondary Optuna study to find optimal blending weights from OOF arrays...")

def objective_3way_blend(trial):
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_cat = trial.suggest_float('w_cat', 0.0, 1.0)
    
    total_w = w_lgb + w_xgb + w_cat
    if total_w == 0: return -1.0
    
    w_lgb, w_xgb, w_cat = w_lgb/total_w, w_xgb/total_w, w_cat/total_w
    
    blended_oof = (w_lgb * oof_preds_lgb) + (w_xgb * oof_preds_xgb) + (w_cat * oof_preds_cat)
    return r2_score(y_train, blended_oof)

study_blend = optuna.create_study(direction="maximize")
study_blend.optimize(objective_3way_blend, n_trials=1000)

raw_lgb = study_blend.best_params['w_lgb']
raw_xgb = study_blend.best_params['w_xgb']
raw_cat = study_blend.best_params['w_cat']
total_raw = raw_lgb + raw_xgb + raw_cat

best_lgb_w = raw_lgb / total_raw
best_xgb_w = raw_xgb / total_raw
best_cat_w = raw_cat / total_raw

print("\n" + "="*50)
print("🎯 OPTIMAL MULTI-MODEL ENSEMBLE WEIGHTS FOUND:")
print(f"LightGBM Weight : {best_lgb_w * 100:.2f}%")
print(f"XGBoost Weight  : {best_xgb_w * 100:.2f}%")
print(f"CatBoost Weight : {best_cat_w * 100:.2f}%")
print(f"🏆 Meta-Optimized 5-Fold Blended CV R2: {study_blend.best_value:.4f}")
print("="*50)

final_ensemble_preds = (best_lgb_w * test_preds_lgb) + (best_xgb_w * test_preds_xgb) + (best_cat_w * test_preds_cat)

Running secondary Optuna study to find optimal blending weights from OOF arrays...

🎯 OPTIMAL MULTI-MODEL ENSEMBLE WEIGHTS FOUND:
LightGBM Weight : 14.43%
XGBoost Weight  : 35.99%
CatBoost Weight : 49.58%
🏆 Meta-Optimized 5-Fold Blended CV R2: 0.9564


In [9]:
print("Aligning final cross-validated ensemble array with output file formatting...")

test_rows_final = df[df['is_train'] == 0].copy()

submission_df = pd.DataFrame({
    'Index': test_rows_final['test_file_id'].astype(int), 
    'demand': final_ensemble_preds
})

submission_df.to_csv('optuna_spatial_temporal_submission.csv', index=False)

print("\n" + "="*60)
print("🎉 Master Ensemble Pipeline Exported Successfully!")
print("Filename: optuna_spatial_temporal_submission.csv")
print(f"Total rows aligned: {len(submission_df)}")
print("="*60)

Aligning final cross-validated ensemble array with output file formatting...

🎉 Master Ensemble Pipeline Exported Successfully!
Filename: optuna_spatial_temporal_submission.csv
Total rows aligned: 41778
